In [9]:
import os, sys, pathlib, importlib
import pandas as pd
import json
import time
from tqdm import tqdm
sys.path.append(os.path.abspath(".."))
from utils import preprocess_images
from utils.preprocess_images import download_images_batch


## clf

In [34]:
from PIL import Image
import torch
from transformers import CLIPProcessor, CLIPModel
from ultralytics import YOLO

# ----- 1. MODELS -----
# YOLOv8 人物检测
yolo_model = YOLO("yolov8n.pt")  # 或 yolov8n-seg 根据需要，n是轻量模型

# CLIP 文本 + 图像
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

## 标签文本
labels = [
    "Professional portrait: ONLY one person, front-facing, occupying most of the image, no meaningful background",
    "Lifestyle photo: person in everyday life or outdoor context, engaged in activities or travel, visible and richer background"
    ]

# ----- 2. DATA -----
# IMG_FOLDER="images\host_pic_sample"
IMG_FOLDER="host_pic_paris_43258"
images=[]

## marche le mieux quand on fournit sys et user prompt à la fois
start_time=time.time()

for filename in os.listdir(IMG_FOLDER):
    if not filename.endswith((".jpg",".JPG",".jpeg",".png","tif")):
        continue #==skip
    img_path=os.path.join(IMG_FOLDER, filename)
    images.append(img_path)
print(f"{len(images)} images")


# ----- 3. Pipeline -----
results = []
SAVE_EVERY=1000
for i, img_path in enumerate(tqdm(images[:1001], desc="classify host profile picture...")):
    image = Image.open(img_path).convert("RGB")
    
    # # YOLO 人物检测
    yolo_preds = yolo_model(source=image, verbose=False)

    # yolo_preds = yolo_model(image,conf=0.12)#调整置信度
    if len(yolo_preds[0].boxes) == 0:
        results.append({
            "image": img_path,
            "host_id":os.path.splitext(os.path.basename(img_path))[0],
            "label": "no_person",
            "confidence": 1.0,
            "bbox": None
        })
        continue

    # CLIP 分类同之前代码
    inputs = clip_processor(text=labels, images=image, return_tensors="pt", padding=True)
    outputs = clip_model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1)
    conf, idx = probs.max(dim=1)
    pred_label = ["pro_style", "life_style"][idx.item()]  # 对应原始分类标签
    
    results.append({
        "image": img_path,
        "host_id":os.path.splitext(os.path.basename(img_path))[0],
        # "label": labels[idx.item()],
        "label":pred_label,
        "confidence": conf.item(),
        "bbox": [box.xyxy.tolist() for box in yolo_preds[0].boxes]  # YOLO检测框
    })
    if i!=0 and i % SAVE_EVERY == 0:# //整除，%取余数
        path_result="host_pic_paris_43258/results_clf.json"
        with open(path_result,'w', encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)            
            print(f"save intermediate results: first {i} / {len(images)} images")
        
end_time=time.time()
print(f"\n [DONE] classify {len(results)} images: {end_time-start_time:.2f} sec! \n")

path_result="host_pic_paris_43258/results_clf.json"
with open(path_result,'w', encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"[SAVE] {len(results)} results saved to {path_result}!\n")
    

43055 images


classify host profile picture...: 100%|██████████| 1001/1001 [02:40<00:00,  6.24it/s]

save intermediate results: first 1000 / 43055 images

 [DONE] classify 1001 images: 160.67 sec! 

[SAVE] 1001 results saved to host_pic_paris_43258/results_clf.json!



In [36]:
## copie-coller
import os
import shutil
from pathlib import Path

# 原始图片根目录
# IMG_FOLDER="host_pic_paris_43258"

with open(path_result,"r", encoding="utf-8")as f:
    results=json.load(f)


# 人工检查目录
IMG_FOLDER_SAMPLE = Path(r"host_pic_sample1000")
os.makedirs(IMG_FOLDER_SAMPLE, exist_ok=True)
# IMG_FOLDER_SAMPLE.mkdir(parents=True, exist_ok=True)

start_time=time.time()

for i, item in enumerate(tqdm(results, desc=f"shutil pic to {IMG_FOLDER_SAMPLE}...")):
    # 统一路径分隔符（防止 \ 和 / 混用）
    src_path = Path(item['image'])
    filename=item["host_id"]
    type=item["label"]
    
    dst_path=os.path.join(IMG_FOLDER_SAMPLE, type,  f"{filename}.jpg")
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)
    
    # shutil
    if src_path.exists():
        shutil.copy2(src_path, dst_path)
    else:
        print("找不到:", src_path)
end_time=time.time()

print(f"[DONE] {i+1} images shutiled in {end_time-start_time:.2f} sec!")


shutil pic to host_pic_sample1000...: 100%|██████████| 1001/1001 [00:01<00:00, 844.90it/s]

[DONE] 1001 images shutiled in 1.19 sec!


In [30]:
dst_path

'host_pic_sample1000\\life_style\\10014012.jpg'

In [22]:
## check results
with open(path_result,"r", encoding="utf-8")as f:
    results=json.load(f)
print(len(results))
print(results[0])

1001
{'image': 'host_pic_paris_43258\\100004988.jpg', 'host_id': '100004988', 'label': 'pro_style', 'confidence': 0.9665592312812805, 'bbox': [[[12.649340629577637, 21.16490364074707, 207.73704528808594, 224.56495666503906]]]}


In [ ]:
# path_result="images\host_pic_sample/results_clf3.json"

import json
with open("images\host_pic_sample/metadata_host_pic.json","r", encoding="utf-8")as f:
    metadata=json.load(f)
with open(path_result,"r", encoding="utf-8")as f:
    results=json.load(f)

import pandas as pd

host_data=[]
for  host in metadata:
    host_id=host["host_id"]
    # print(type(host_id))
    pred_label=[res['label'] for res in results if res['host_id']==str(host_id)][0]
    true_label=host["true_label"]
    host_data.append({"host_id":host_id,
                      'true_label':true_label,
                      'pred_label':pred_label}
                     )
    # print(f"host_id:{host_id}, true label:{true_label}, pred label:{pred_label} \n")
    # break
host_data_df=pd.DataFrame(host_data)
display(host_data_df)

host_data_df['tp']=host_data_df.apply(lambda row : 1 if str(row["true_label"])==str(row["pred_label"]) else 0, axis=1)
print(host_data_df['tp'].mean())

,host_id,true_label,pred_label
0,102571900,pro_style,pro_style
1,106294215,life_style,no_person
2,106365215,life_style,life_style
3,137154154,life_style,life_style
4,212791574,no_person,no_person
5,2379345,no_person,no_person
6,24654560,life_style,life_style
7,2798386,pro_style,pro_style
8,28470251,pro_style,pro_style
9,32741638,pro_style,no_person


0.8


## download pics

In [5]:
import pandas as pd
df_all=pd.read_csv("../data_all\listings_paris_london2406.csv")
print(df_all.shape)
print(df_all.host_id.value_counts(dropna=False))

C:\Users\yeliu\AppData\Local\Temp\ipykernel_15548\674656137.py:2: DtypeWarning: Columns (68,77,79) have mixed types. Specify dtype option on import or set low_memory=False.
  df_all=pd.read_csv("../data_all\listings_paris_london2406.csv")


(112594, 129)
host_id
33889201     713
50502817     319
314994947    299
314162972    298
50978178     290
            ... 
46032227       1
353642626      1
127644889      1
524232082      1
509027540      1
Name: count, Length: 72607, dtype: int64


In [7]:
print(df_all[["host_id", 'city']].value_counts(dropna=False))


host_id    city  
33889201   paris     459
50502817   paris     319
314994947  paris     299
314162972  london    298
50978178   paris     290
                    ... 
33911767   paris       1
33913374   london      1
33915626   paris       1
33916587   london      1
583113536  london      1
Name: count, Length: 72636, dtype: int64


In [ ]:
df_pic=df_all.drop_duplicates(subset='host_picture_url')
df_pic_paris=df_pic[df_pic['city']=="paris"]
df_pic_london=df_pic[df_pic['city']=="london"]

print(df_pic.shape, df_pic_paris.shape, df_pic_london.shape)

df_pic_paris[['host_id','host_picture_url']].value_counts(dropna=False, ascending=False)

(68309, 129) (43258, 129) (25051, 129)


host_id    host_picture_url                                                                                                                      
2626       https://a0.muscache.com/im/pictures/user/ad6a9447-d6fa-4b6b-a820-1c5b52cd5359.jpg?aki_policy=profile_x_medium                             1
141709495  https://a0.muscache.com/im/pictures/user/User-141709495/original/286fd427-b526-4e5b-9552-0bcdd2f3ccf3.jpeg?aki_policy=profile_x_medium    1
141594292  https://a0.muscache.com/im/pictures/user/User-141594292/original/a29064c3-bf0d-4c33-89ec-72540f9432f6.jpeg?aki_policy=profile_x_medium    1
141647542  https://a0.muscache.com/im/pictures/user/ae9a1f66-addc-4473-8493-bbb16985415d.jpg?aki_policy=profile_x_medium                             1
141657670  https://a0.muscache.com/im/pictures/user/User-141657670/original/f64beb0d-f717-4f1c-9177-37c10d8858d5.jpeg?aki_policy=profile_x_medium    1
                                                                                                   

In [4]:
df_pic_paris_sample=df_pic_paris[:10]
# display(df_pic_paris_sample)
# id_url_list: list of tuples [(host_id, url), ...]
id_url_list=[(row['host_id'], row["host_picture_url"]) for _, row in df_pic_paris_sample.iterrows()]
print(id_url_list)

[(3631, 'https://a0.muscache.com/im/users/3631/profile_pic/1375800198/original.jpg?aki_policy=profile_x_medium'), (433758, 'https://a0.muscache.com/im/users/433758/profile_pic/1330955021/original.jpg?aki_policy=profile_x_medium'), (7903, 'https://a0.muscache.com/im/users/7903/profile_pic/1280002723/original.jpg?aki_policy=profile_x_medium'), (2626, 'https://a0.muscache.com/im/pictures/user/ad6a9447-d6fa-4b6b-a820-1c5b52cd5359.jpg?aki_policy=profile_x_medium'), (228508, 'https://a0.muscache.com/im/users/228508/profile_pic/1284123543/original.jpg?aki_policy=profile_x_medium'), (22155, 'https://a0.muscache.com/im/pictures/user/975c3587-7e98-490e-a35e-b39ed09b8355.jpg?aki_policy=profile_x_medium'), (28422, 'https://a0.muscache.com/im/users/28422/profile_pic/1319547089/original.jpg?aki_policy=profile_x_medium'), (33534, 'https://a0.muscache.com/im/pictures/user/4f775bb4-9080-4943-aa57-d516708aec81.jpg?aki_policy=profile_x_medium'), (37107, 'https://a0.muscache.com/im/users/37107/profile_pic

## download host pic pariss

In [5]:
importlib.reload(preprocess_images)
from utils.preprocess_images import download_images_batch
download_images_batch(df_pic_paris, out_dir='host_pic_paris_43258', max_workers=12)


len df:43258!


[WARNING] 无效 URL: https://a0.muscache.com/im/users/295001/profile_pic/1371196555/original.jpg?aki_policy=profile_x_medium


[WARNING] 无效 URL: https://a0.muscache.com/im/pictures/user/4693c6a2-64c1-44df-9cdf-0b6e93507f7f.jpg?aki_policy=profile_x_medium


[WARNING] 无效 URL: https://a0.muscache.com/im/users/5947575/profile_pic/1366192247/original.jpg?aki_policy=profile_x_medium


[WARNING] 无效 URL: https://a0.muscache.com/im/users/7327717/profile_pic/1373194686/original.jpg?aki_policy=profile_x_medium


...

[WARNING] 无效 URL: https://a0.muscache.com/im/pictures/user/User-523953429/original/6245003f-1f44-49c3-b7d7-6587e40b28b2.jpeg?aki_policy=profile_x_medium


[WARNING] 无效 URL: https://a0.muscache.com/im/pictures/user/User-575732047/original/7a107cae-1fea-40c8-bdb8-6c67a730d43c.jpeg?aki_policy=profile_x_medium


[WARNING] 无效 URL: https://a0.muscache.com/im/users/9515983/profile_pic/1383777330/original.jpg?aki_policy=profile_x_medium



[SUCCES] downloaded 43258 pics : 7327.01 sec!

success: 43052 pics!
fails: 206 pics!


{22155: 'host_pic_paris_43258\\22155.jpg',
 439130: 'host_pic_paris_43258\\439130.jpg',
 228508: 'host_pic_paris_43258\\228508.jpg',
 7903: 'host_pic_paris_43258\\7903.jpg',
 433758: 'host_pic_paris_43258\\433758.jpg',
 3631: 'host_pic_paris_43258\\3631.jpg',
 37107: 'host_pic_paris_43258\\37107.jpg',
 28422: 'host_pic_paris_43258\\28422.jpg',
 429406: 'host_pic_paris_43258\\429406.jpg',
 41718: 'host_pic_paris_43258\\41718.jpg',
 2626: 'host_pic_paris_43258\\2626.jpg',
 33534: 'host_pic_paris_43258\\33534.jpg',
 64627: 'host_pic_paris_43258\\64627.jpg',
 48733: 'host_pic_paris_43258\\48733.jpg',
 44444: 'host_pic_paris_43258\\44444.jpg',
 152242: 'host_pic_paris_43258\\152242.jpg',
 296615: 'host_pic_paris_43258\\296615.jpg',
 64055: 'host_pic_paris_43258\\64055.jpg',
 79843: 'host_pic_paris_43258\\79843.jpg',
 72713: 'host_pic_paris_43258\\72713.jpg',
 464019: 'host_pic_paris_43258\\464019.jpg',
 69389: 'host_pic_paris_43258\\69389.jpg',
 276063: 'host_pic_paris_43258\\276063.jpg',
 

## download host pic london

In [ ]:
# df_pic_london_sample=df_pic_london[:10]
id_url_list=[(row['host_id'], row["host_picture_url"]) for _, row in df_pic_london.iterrows()]
print(len(id_url_list))
print(id_url_list[0])


25051
(517837625, 'https://a0.muscache.com/im/pictures/user/d4354a96-266f-4734-b69b-3b66e1de09fa.jpg?aki_policy=profile_x_medium')


In [ ]:
importlib.reload(preprocess_images)
from utils.preprocess_images import download_images_batch
download_images_batch(df_pic_london, out_dir='images/host_pic_london_25051', max_workers=12)


len df:25051!


[WARNING] 无效 URL: https://a0.muscache.com/im/pictures/user/a53dbc41-91f2-4353-9421-75640dbf0f9a.jpg?aki_policy=profile_x_medium


[WARNING] 无效 URL: https://a0.muscache.com/im/pictures/user/User-558720586/original/6055dfd9-0d3c-4767-845d-0139ae1158e2.jpeg?aki_policy=profile_x_medium


[WARNING] 无效 URL: https://a0.muscache.com/im/pictures/user/9868ef0e-4794-410b-bd23-cb04b908ed4a.jpg?aki_policy=profile_x_medium


[WARNING] 无效 URL: https://a0.muscache.com/im/users/13278656/profile_pic/1395186363/original.jpg?aki_policy=profile_x_medium


...

[WARNING] 无效 URL: https://a0.muscache.com/im/pictures/user/User-200196351/original/3e322f9b-d939-47b1-bd07-870103d2a710.jpeg?aki_policy=profile_x_medium


[WARNING] 无效 URL: https://a0.muscache.com/im/pictures/user/0297550f-1a3f-4d67-a4af-f70c9a87fc9f.jpg?aki_policy=profile_x_medium



[SUCCES] downloaded 25051 pics : 5551.15 sec!

success: 24819 pics!
fails: 232 pics!


{456550337: 'images/host_pic_london_25051\\456550337.jpg',
 483861278: 'images/host_pic_london_25051\\483861278.jpg',
 102091934: 'images/host_pic_london_25051\\102091934.jpg',
 517837625: 'images/host_pic_london_25051\\517837625.jpg',
 30949469: 'images/host_pic_london_25051\\30949469.jpg',
 487206988: 'images/host_pic_london_25051\\487206988.jpg',
 28880140: 'images/host_pic_london_25051\\28880140.jpg',
 551207872: 'images/host_pic_london_25051\\551207872.jpg',
 483663228: 'images/host_pic_london_25051\\483663228.jpg',
 303622628: 'images/host_pic_london_25051\\303622628.jpg',
 19896713: 'images/host_pic_london_25051\\19896713.jpg',
 403509869: 'images/host_pic_london_25051\\403509869.jpg',
 130940430: 'images/host_pic_london_25051\\130940430.jpg',
 175222658: 'images/host_pic_london_25051\\175222658.jpg',
 252303991: 'images/host_pic_london_25051\\252303991.jpg',
 2902129: 'images/host_pic_london_25051\\2902129.jpg',
 495977998: 'images/host_pic_london_25051\\495977998.jpg',
 448994

: 